In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import torch
from tokenizer import Tokenizer
from model import Model

In [2]:
my_tokenizer = Tokenizer("tokenizer.json")
prompt = "Merhaba, senin adın ne söyleyebilir misin?"
tokens = my_tokenizer.encode(prompt)
torch.manual_seed(2)
model = Model(len(my_tokenizer.vocab), embedding_dim=4)
model(tokens)  # batch dimension ekle

tensor([[[-0.5483,  0.5117, -0.4988,  ..., -0.0244,  0.1020, -0.7818],
         [-0.5603,  0.5173, -0.4947,  ..., -0.0288,  0.1046, -0.7888],
         [-0.5531,  0.5156, -0.4956,  ..., -0.0269,  0.1046, -0.7847],
         ...,
         [-0.5493,  0.5125, -0.4981,  ..., -0.0249,  0.1025, -0.7824],
         [-0.5567,  0.5163, -0.4953,  ..., -0.0278,  0.1045, -0.7867],
         [-0.5522,  0.5148, -0.4962,  ..., -0.0264,  0.1041, -0.7841]]],
       grad_fn=<ViewBackward0>)

In [3]:
model


Model(
  (embedding): Embedding(1376, 4)
  (layers): Sequential(
    (0): DecoderBlock(
      (attention): MultiHeadAttention(
        (num_heads): ModuleList(
          (0-1): 2 x CausalAttention(
            (q_weights): Linear(in_features=4, out_features=2, bias=False)
            (k_weights): Linear(in_features=4, out_features=2, bias=False)
            (v_weights): Linear(in_features=4, out_features=2, bias=False)
            (dropout): Dropout(p=0, inplace=False)
          )
        )
        (projection): Linear(in_features=4, out_features=4, bias=True)
      )
      (layer_norm1): LayerNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=4, out_features=16, bias=True)
        (up_proj): Linear(in_features=4, out_features=16, bias=True)
        (down_proj): Linear(in_features=16, out_features=4, bias=True)
        (gelu): GELU()
      )
      (layer_norm2): LayerNorm()
    )
    (1): DecoderBlock(
      (attention): MultiHeadAttention(
        (num_heads): ModuleList

In [6]:
all_similarities= torch.zeros((sentence_meaning.shape[0],sentence_meaning.shape[0]))

for i in range(sentence_meaning.shape[0]):
    similarities = []
    for j in range(sentence_meaning.shape[0]):
        sim = sentence_meaning[i][0]*sentence_meaning[j][0]+sentence_meaning[i][1]*sentence_meaning[j][1]+sentence_meaning[i][2]*sentence_meaning[j][2]+sentence_meaning[i][3]*sentence_meaning[j][3]
        similarities.append(sim)
    all_similarities[i] = torch.tensor(similarities)
all_similarities.detach().numpy()

NameError: name 'sentence_meaning' is not defined

In [6]:
sentence_meaning= model(tokens)

In [15]:
all_similarities= torch.zeros((sentence_meaning.shape[0],sentence_meaning.shape[0]))
sentence_meaning=sentence_meaning.squeeze(0)
for i in range(sentence_meaning.shape[0]):
    i_similarities = torch.zeros(sentence_meaning.shape[0])
    for j in range(sentence_meaning.shape[0]):
        for k in range(sentence_meaning.shape[1]):
            i_similarities[j] += sentence_meaning[i][k]*sentence_meaning[j][k]
    all_similarities[i] = i_similarities
all_similarities.detach().numpy()

RuntimeError: The expanded size of the tensor (1) must match the existing size (24) at non-singleton dimension 0.  Target sizes: [1].  Tensor sizes: [24]

In [8]:
# query, key, value yaklaşımıyla benzerlik hesaplama
sentence_meaning=sentence_meaning.squeeze(0)

all_similarities=sentence_meaning @ sentence_meaning.T # her bir kelime diğer kelimelerle olan benzerlikleri gösterir



In [10]:
attention_weights= torch.softmax(all_similarities, dim=-1) # benzerlikleri normalleştir

In [11]:
# yukarıda her bir kelime diğer kelimelere ne kadar benziyoru gördük şimdi de her bir kelime bu cümle içinde ne kadar önemli

attention_weights = torch.softmax(all_similarities, dim=1)
attention_weights

tensor([[0.0414, 0.0418, 0.0414, 0.0417, 0.0417, 0.0417, 0.0418, 0.0419, 0.0417,
         0.0417, 0.0418, 0.0418, 0.0419, 0.0415, 0.0414, 0.0417, 0.0413, 0.0419,
         0.0416, 0.0418, 0.0418, 0.0417, 0.0416, 0.0414],
        [0.0411, 0.0419, 0.0410, 0.0418, 0.0419, 0.0417, 0.0420, 0.0422, 0.0416,
         0.0416, 0.0421, 0.0421, 0.0421, 0.0413, 0.0412, 0.0416, 0.0409, 0.0422,
         0.0416, 0.0418, 0.0420, 0.0417, 0.0416, 0.0411],
        [0.0400, 0.0404, 0.0425, 0.0418, 0.0419, 0.0429, 0.0419, 0.0419, 0.0412,
         0.0413, 0.0425, 0.0409, 0.0419, 0.0420, 0.0404, 0.0405, 0.0407, 0.0440,
         0.0435, 0.0445, 0.0414, 0.0403, 0.0407, 0.0409],
        [0.0403, 0.0410, 0.0417, 0.0419, 0.0419, 0.0424, 0.0420, 0.0422, 0.0414,
         0.0414, 0.0424, 0.0415, 0.0421, 0.0416, 0.0405, 0.0410, 0.0406, 0.0434,
         0.0427, 0.0435, 0.0418, 0.0409, 0.0410, 0.0409],
        [0.0402, 0.0411, 0.0417, 0.0418, 0.0421, 0.0425, 0.0419, 0.0420, 0.0414,
         0.0412, 0.0426, 0.0416, 0.0420

In [11]:
sentence_context_vectors = attention_weights @ sentence_meaning
sentence_context_vectors

tensor([[-0.5688,  0.5069, -0.5050,  ..., -0.0256,  0.0932, -0.7931],
        [-0.5691,  0.5067, -0.5052,  ..., -0.0256,  0.0930, -0.7933],
        [-0.5688,  0.5069, -0.5050,  ..., -0.0256,  0.0932, -0.7931],
        ...,
        [-0.5688,  0.5069, -0.5050,  ..., -0.0256,  0.0932, -0.7931],
        [-0.5690,  0.5068, -0.5051,  ..., -0.0256,  0.0931, -0.7932],
        [-0.5688,  0.5069, -0.5050,  ..., -0.0256,  0.0932, -0.7931]],
       grad_fn=<MmBackward0>)

In [ ]:
"""u_tokeneizer= Tokenizer("tokenizer.json")
tokens = u_tokeneizer.encode("Merhaba, senin adın ne söyleyebilir misin?")
torch.manual_seed(2)
model= Model(len(u_tokeneizer.vocab), embedding_dim=4,device='cpu')
sentence_meaning= model(tokens)

attention_weights = torch.softmax(sentence_meaning @ sentence_meaning.T, dim=1)
sentence_context_vectors = attention_weights @ sentence_meaning"""

'u_tokeneizer= Tokenizer("tokenizer.json")\ntokens = u_tokeneizer.encode("Merhaba, senin adın ne söyleyebilir misin?")\ntorch.manual_seed(2)\nmodel= Model(len(u_tokeneizer.vocab), embedding_dim=4,device=\'cpu\')\nsentence_meaning= model(tokens)\n\nattention_weights = torch.softmax(sentence_meaning @ sentence_meaning.T, dim=1)\nsentence_context_vectors = attention_weights @ sentence_meaning'

In [15]:
q_weights = torch.nn.Linear(1376, 3,bias=False) # 4 boyutlu embeddingi 3 boyuta indiriyoruz
k_weights = torch.nn.Linear(1376, 3,bias=False)
v_weights = torch.nn.Linear(1376, 3,bias=False)


q_of_sentence = q_weights(sentence_meaning)
k_of_sentence = k_weights(sentence_meaning)
v_of_sentence = v_weights(sentence_meaning)

q_of_sentence.shape, k_of_sentence.shape, v_of_sentence.shape

(torch.Size([24, 3]), torch.Size([24, 3]), torch.Size([24, 3]))

In [ ]:
"""from plot_tokens import plot_tokens

u_sentences =[
  {  "words":q_of_sentence.detach().numpy(),
    "labels":my_tokenizer.tokenize(prompt),
    "color":"blue",},
    {
    "words":k_of_sentence.detach().numpy(),
    "labels":my_tokenizer.tokenize(prompt),
    "color":"red",
    },
    {
    "words":v_of_sentence.detach().numpy(),
    "labels":my_tokenizer.tokenize(prompt),
    "color":"green",
    }
]
plot_tokens(u_sentences, title="Query, Key, Value")"""

'from plot_tokens import plot_tokens\n\nu_sentences =[\n  {  "words":q_of_sentence.detach().numpy(),\n    "labels":my_tokenizer.tokenize(prompt),\n    "color":"blue",},\n    {\n    "words":k_of_sentence.detach().numpy(),\n    "labels":my_tokenizer.tokenize(prompt),\n    "color":"red",\n    },\n    {\n    "words":v_of_sentence.detach().numpy(),\n    "labels":my_tokenizer.tokenize(prompt),\n    "color":"green",\n    }\n]\nplot_tokens(u_sentences, title="Query, Key, Value")'

In [16]:
#şimdi query ile key arasında benzerlik hesaplayalım

k_of_sentence.shape[-1] # son boyut aynı olmalı

3

In [17]:
attention_scores = q_of_sentence @ k_of_sentence.T
attention_weights = torch.softmax(attention_scores / torch.sqrt(torch.tensor(k_of_sentence.shape[-1], dtype=torch.float32)), dim=1)

In [18]:
context_vector= attention_weights @ v_of_sentence
context_vector  # artık burda bir bağlam ve bilgi var.
# sözlük bilgisini embedding tarafında asıl bağlam da burda tutulur

tensor([[-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024],
        [-0.1573, -0.0651,  0.1024]], grad_fn=<MmBackward0>)

In [19]:
# causal self attention
# sadece geçmişteki kelimelere bakarak benzerlik hesaplama

mask= torch.tril(torch.ones((attention_weights.shape[0],attention_weights.shape[1])))  # alt üçgen maske oluşturma
#torch.softmax(mask*attention_weights,dim=1) #burda o yapınca softmaxte bir değere karşılık geliyor. Ama biz maskeleme yapmak istediğimizden dolayı spoftmaxten çıktısınında 0 olmasını istediğimden 0'ların hepsini - sonsuz yapıyorum

masked_attention_weights= attention_weights.masked_fill(mask==0, float('-inf'))
softmaxe_attention_weights= torch.softmax(masked_attention_weights, dim=1)
# bu aşamadan sonra bazı durumlarda baskın olabilecek hücreler olabiliyor. Bunlar olmasın diye dropout işlemiyle rastgele kapatabiliyoruz. Bu sayede eğitim aşamasında herhangi bir şeye odaklanmasının önüne geçiyoruz rastgele tahmin yapmasını sağlıyoruz.


In [20]:
dropout_rate= 0.5 # parametrelerin yüzde 50sini dropout yap
torch.manual_seed(42)
dropout= torch.nn.Dropout(dropout_rate)
dropout(softmaxe_attention_weights)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6667, 0.6667, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4000, 0.0000, 0.4000, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000

In [ ]:
from türkçe_dil_modelim.multi_head_attention_old import MultiHeadAttention
import torch
import torch.nn as nn
    
multi_head_attention = MultiHeadAttention(embed_size=4, output_size=8, num_heads=2, context_length=10, dropout_rate=0.5)
multi_head_attention(torch.randn(4,4)).shape


torch.Size([4, 8])

In [4]:
import torch 
import torch.nn as nn
nn.Parameter(torch.ones(4,4))

Parameter containing:
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]], requires_grad=True)

In [12]:
ex_tensor= torch.tensor([[1,2,3],[4,5,6]], dtype=torch.float32)
means= ex_tensor.mean(dim=-1, keepdim=True)  # sütun bazında ortalama alır
stds= ex_tensor.std(dim=-1, keepdim=True,unbiased=False)    # sütun bazında standart sapma alır
means, stds, ex_tensor

(tensor([[2.],
         [5.]]),
 tensor([[0.8165],
         [0.8165]]),
 tensor([[1., 2., 3.],
         [4., 5., 6.]]))

In [ ]:
import torch 
import torch.nn as nn


class GELU(nn.Module):
    def __init__(self):
        super().__init__()


    def forward(self,x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3)))) # aktivasyon fonksiyonunun amacı değerleri duruma bağlı olarak belli aralıkların içinde aktive etmektir.

In [ ]:
gelu= GELU()
gelu(torch.tensor([[-1.0, 0.0, 1.0],[3,1.5,0.5]])) 

tensor([[-0.1588,  0.0000,  0.8412],
        [ 2.9964,  1.3996,  0.3457]])

In [14]:
import torch.nn.functional as F

F.gelu(torch.tensor([[-1.0, 0.0, 1.0],[3,1.5,0.5]]),approximate="tanh") #direkt torchtan da bu şekilde çağrılabilir

tensor([[-0.1588,  0.0000,  0.8412],
        [ 2.9964,  1.3996,  0.3457]])

In [ ]:
import torch
import torch.nn as nn

#başka modellerde araştırma yaparken Parameter ifadesini gördüğünde eğitilen bir şeyb vardır


class MLP(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()

        self.gate_proj= nn.Linear(embedding_dim, hidden_dim)
        self.up_proj= nn.Linear(embedding_dim, hidden_dim)
        self.down_proj= nn.Linear(hidden_dim, embedding_dim)
        self.gelu= GELU()
    def forward(self,x):
        gate= self.gate_proj(x)
        gate= self.gelu(gate)
        up= self.up_proj(x)
        fuse= gate*up
        outputs= self.down_proj(fuse)
        return outputs
    

In [15]:
import torch.nn as nn
from multi_head_attention import MultiHeadAttention
from layer_norm import LayerNorm
from mlp import MLP

class DecoderBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, context_length):
        super().__init__()
        
        self.self_attention= MultiHeadAttention(embed_size=embedding_dim, output_size=embedding_dim, num_heads=num_heads, context_length=context_length)
        self.norm1= LayerNorm(embedding_dim)
        self.mlp= MLP(embedding_dim, hidden_dim=embedding_dim*4)
        self.norm2= LayerNorm(embedding_dim)

    def forward(self,x):
        res= self.norm1(x)

        x= self.self_attention(x)
        x= self.norm2(x)
        x= x+res

        res=x
        res= self.norm1(res)
        x= self.mlp(x)
        x= self.norm2(x)
        x= x+res
        return x


In [16]:
example_input= torch.tensor([[1,2,3,4],[5,6,7,8]], dtype=torch.float32)
decoder_block= DecoderBlock(embedding_dim=4, num_heads=2, context_length=3)
decoder_block(example_input)

tensor([[-0.0691, -0.1170,  0.3164, -0.1303],
        [-0.1339, -0.1842,  0.5993, -0.2811]], grad_fn=<AddBackward0>)

In [33]:
output= model(tokens)
torch.softmax(output[0][0], dim=-1) # her bir hücre sözlükteki kelimelere karşılık gelen olasılıkları gösterir

max_prob, max_index= torch.max(torch.softmax(output[0][0], dim=-1), dim=-1)
max_prob, max_index

(tensor(0.0030, grad_fn=<MaxBackward0>), tensor(652))

In [34]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 54.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 13.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.6/803.6 kB 30.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [transformers] [transformers]ub]

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [35]:
from transformers import AutoTokenizer, AutoModelForCausalLM


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [38]:
q_tokenizer= AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
q_model= AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
q_encode= q_tokenizer.encode("Merhaba, senin adın ne söyleyebilir misin?", return_tensors="pt")
q_tokens = q_tokenizer.tokenize("Merhaba, senin adın ne söyleyebilir misin?")
q_tokens,q_encode # qwen modelinin kelimeleri tokenleştirmesiyle çıkan tokenler

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 356.20it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


(['Mer',
  'hab',
  'a',
  ',',
  'Ġsen',
  'in',
  'Ġad',
  'Ä±n',
  'Ġne',
  'ĠsÃ¶yle',
  'y',
  'eb',
  'il',
  'ir',
  'Ġmis',
  'in',
  '?'],
 tensor([[ 26716,  10573,     64,     11,   6124,    258,    993,  16116,    834,
          129284,     88,   3065,    321,    404,   5786,    258,     30]]))